# Phase 2 — Mask Improvement V4
Select a GPU runtime. Train on 56 actually approved training-mask updates; preserve all 40 original validation references for comparison. Existing releases and results remain available. No classifier retraining or test rerun occurs here. A >90% IoU is a measured target, not guaranteed.

## 1. Setup

In [ ]:
from google.colab import drive
from pathlib import Path
import sys, subprocess, hashlib, urllib.request, torch

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/CameraTrap_Phase2')
CODE = Path('/content/mask_training_v4_code'); CODE.mkdir(exist_ok=True)
CACHE = Path('/content/mask_training_v4_cache')
INPUTS = {'run.py': {'path': 'experiments/phase2_v4_masks/run.py', 'sha256': '3a1a06f67303ad0eba2623cff9693e18f016ec71e98f5277b8cd078d8472403b'}, 'optimize_v3.py': {'path': 'experiments/phase2_v3/optimize.py', 'sha256': '7624974ae0d83c56e82ccd24d025353f47f7dd10a4893fa42933f11a3660ecb3'}, 'Reviewed_Mask_Overrides_V4.zip': {'path': 'results/phase2/data_review_v4/human_review_round3_20261120/Reviewed_Mask_Overrides_V4.zip', 'sha256': '44d378c35d9b5d10efc2687866797d2f504ad5da543fc5b2b575431e8f2ca292'}}
for name, record in INPUTS.items():
    path = CODE/name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != record['sha256']:
        print('Downloading:', name, flush=True)
        with urllib.request.urlopen('https://raw.githubusercontent.com/salehdev3/Capstone_5/01fc892/'+record['path'], timeout=60) as response:
            source = response.read()
        assert hashlib.sha256(source).hexdigest() == record['sha256'], name
        path.write_bytes(source)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.3.221'], check=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not connected; training requires GPU')
def stage(name):
    print('Starting:', name, flush=True)
    subprocess.run([sys.executable, '-u', str(CODE/'run.py'), '--root', str(ROOT), '--cache', str(CACHE), '--code', str(CODE), '--stage', name], check=True)


## 2. Prepare reviewed training data
Verify pixels and source hashes, apply 56 training overrides in local cache, and copy validation labels unchanged. The unresolved validation image remains included. After a runtime reset, repeat setup and preparation before resuming training.

In [ ]:
stage('prepare')


## 3. Continue segmentation training
Warm-start the V3 best YOLO11s-seg checkpoint, low learning rate, up to 40 epochs with early stopping and resumable Drive checkpoints.

In [ ]:
stage('train')


## 4. Compare on unchanged validation
Compare standalone models, then the previous full mask pipeline and SAM2.1-t alternatives. Inputs use detector boxes/learned points, never human boxes as inference prompts. Human references are used for scoring only. The CPU runtime/deployment audit remains a separate release check.

In [ ]:
stage('compare')


## 5. Download measured results
Includes actual logs, plots, selected validation policies and provenance. Weights remain in Drive.

In [ ]:
import zipfile
from google.colab import files
RUN = ROOT/'phase2_results/mask_training_v4'
archive = RUN/'Mask_Improvement_V4_Review.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(RUN.rglob('*')):
        if path.is_file() and path.suffix in ('.json','.csv','.png','.jpg','.yaml','.md'):
            z.write(path, path.relative_to(RUN))
print('Saved:', archive)
files.download(str(archive))
